In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, average_precision_score

from xgboost import XGBClassifier
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


In [2]:
CLEANED_FEATURE_FILE = PROJECT_ROOT / "artifacts/dashboard/cleaned_ieee_features.csv"
if not CLEANED_FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Missing prepared IEEE-CIS feature file: {CLEANED_FEATURE_FILE}. "
        "Export the cleaned feature matrix before running this comparison notebook."
    )

df = pd.read_csv(CLEANED_FEATURE_FILE)

y = df["isFraud"]
X = df.drop("isFraud", axis=1)


In [3]:
from sklearn.model_selection import train_test_split

# تقسيم الداتا مع ضمان توازن الفئات
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # دي أهم خطوة عشان الـ Recall
)

In [4]:
print("NaN in Train:", X_train.isnull().sum().sum())
print("NaN in Validation:", X_val.isnull().sum().sum())

NaN in Train: 54618774
NaN in Validation: 13684989


In [5]:
from xgboost import XGBClassifier

# حساب الوزن: (عدد الحالات السليمة / عدد حالات الاحتيال)
ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1])

# تعريف الموديل
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=ratio, # ده السحر اللي هيرفع الـ Recall
    use_label_encoder=False,
    eval_metric='aucpr' # خليه يركز على الـ Area Under PR Curve أثناء التعلم
)

model.fit(X_train, y_train)

/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [19:52:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=None, num_parallel_tree=None, ...)

In [6]:
import plotly.graph_objects as go
from sklearn.metrics import precision_recall_curve, auc

y_probs = model.predict_proba(X_val)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, y_probs)
pr_auc = auc(recall, precision)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=recall, 
    y=precision,
    mode='lines',
    name=f'XGBoost (AUC={pr_auc:.3f})',
    # بنعرض الـ Threshold عند كل نقطة عشان تقرر هتستخدم أنهي واحد
    text=[f"Threshold: {round(t, 3)}" for t in thresholds],
    hovertemplate="<b>Recall: %{x:.3f}</b><br>Precision: %{y:.3f}<br>%{text}<extra></extra>"
))

fig.update_layout(
    title=f"Precision-Recall Curve (Interactive) - PR-AUC: {pr_auc:.4f}",
    xaxis_title="Recall )",
    yaxis_title="Precision",
    hovermode="closest",
    template="plotly_dark"
)

fig.show()

In [7]:
best_threshold = 0.2 
y_pred_final = (y_probs >= best_threshold).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_val, y_pred_final))

              precision    recall  f1-score   support

           0       1.00      0.69      0.81    113975
           1       0.10      0.95      0.18      4133

    accuracy                           0.69    118108
   macro avg       0.55      0.82      0.50    118108
weighted avg       0.97      0.69      0.79    118108



LGBMC

In [8]:
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score

# تعريف الموديل مع تفعيل خاصية التوازن
lgbm_model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    is_unbalance=True, # ده السحر بتاع التوازن في LightGBM
    extra_trees=True,   # بيقلل الـ Overfitting وبيحسن الـ Precision
    random_state=42
)

lgbm_model.fit(X_train, y_train)

# حساب الـ PR-AUC للمقارنة
y_probs_lgbm = lgbm_model.predict_proba(X_val)[:, 1]
lgbm_pr_auc = average_precision_score(y_val, y_probs_lgbm)
print(f"LightGBM PR-AUC: {lgbm_pr_auc:.4f}")

[LightGBM] [Info] Number of positive: 16530, number of negative: 455902


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.090377 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 33349
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 357
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101


LightGBM PR-AUC: 0.7216


In [9]:
from sklearn.metrics import recall_score

# 1. توقع الكلاسات بناءً على Threshold معين (مثلاً 0.4)
y_probs = lgbm_model.predict_proba(X_val)[:, 1]
y_pred = (y_probs >= 0.4).astype(int) 

# 2. حساب الـ Recall
rec = recall_score(y_val, y_pred)

print(f"Recall Score: {rec:.4f}")
print(f"Recall Percentage: {rec*100:.2f}%")

Recall Score: 0.8814
Recall Percentage: 88.14%


In [10]:
from sklearn.metrics import classification_report

print("Classification Report for LightGBM:")
print(classification_report(y_val, y_pred))

Classification Report for LightGBM:
              precision    recall  f1-score   support

           0       1.00      0.89      0.94    113975
           1       0.23      0.88      0.36      4133

    accuracy                           0.89    118108
   macro avg       0.61      0.89      0.65    118108
weighted avg       0.97      0.89      0.92    118108



In [11]:
import plotly.graph_objects as go
from sklearn.metrics import precision_recall_curve, auc

# 1. الحصول على الاحتمالات (Probabilities)
# تأكد إنك سميت الموديل lgbm_model أو غير الاسم حسب الكود عندك
y_probs_lgbm = lgbm_model.predict_proba(X_val)[:, 1]

# 2. حساب قيم الكيرف
precision, recall, thresholds = precision_recall_curve(y_val, y_probs_lgbm)
pr_auc = auc(recall, precision)

# 3. رسم الكيرف التفاعلي باستخدام Plotly
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=recall, 
    y=precision,
    mode='lines',
    name=f'LightGBM (PR-AUC: {pr_auc:.4f})',
    # بنعرض الـ Threshold عند كل نقطة عشان تختار الصح
    text=[f"Threshold: {t:.4f}" for t in thresholds],
    hovertemplate="<b>Recall: %{x:.3f}</b><br>" +
                  "<b>Precision: %{y:.3f}</b><br>" +
                  "%{text}<extra></extra>"
))

fig.update_layout(
    title="Interactive Precision-Recall Curve (LightGBM)",
    xaxis_title="Recall  Fraud)",
    yaxis_title="Precision Fraud)",
    hovermode="closest",
    width=800,
    height=600,
    template="plotly_dark" # أو plotly_dark لو بتحب الخلفية الغامقة
)

fig.show()